# Assignment 1: Gaussians, Categories, and Clusters — SOLUTION KEY (Python)

**Instructor answer key — do not distribute to students.**

Worked solution for the Python (no-GenJAX) stencil `clusters_python.ipynb`.
The numpy cells are solved, and the optional "Now in GenJAX" cells are also
solved so the key demonstrates both paths.

**Corresponding textbook chapters:** Tutorial 1 Ch 5 (Bayesian inference) and
[Tutorial 3 Ch 5 — Mixture Models](https://josephausterweil.github.io/probintro/intro2/05_mixture_models/).

## Setup

Run the cells below to install (if needed) and import everything. The GenJAX install only matters if you plan to do the optional GenJAX cells.


In [ ]:
# Optional: only needed if you do the "Now in GenJAX" cells.
# In Google Colab, uncomment the line below on first run:
# !pip install genjax


In [ ]:
# Standard imports for the numpy-only path.
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

np.random.seed(42)

# GenJAX imports (only needed for the optional "Now in GenJAX" cells).
# If you skip those cells, you can ignore an ImportError here.
# NOTE on Bernoulli: GenJAX has BOTH `bernoulli(logit)` and `flip(probability)`.
# We always use `flip` for this assignment because we want to pass probabilities
# directly. `bernoulli(0.7)` would produce True only ~67% of the time, not 70%.
# NOTE on dtypes: GenJAX distributions produce float32. When you pass scalars
# into a GenJAX model, cast them with jnp.float32(...) so they don't mix with
# numpy's float64 (a dtype mismatch raises a TypeError deep inside the sampler).
try:
    import jax
    import jax.numpy as jnp
    import jax.random as random
    from genjax import gen, normal, flip, ChoiceMap
    key = random.PRNGKey(42)
    _GENJAX_AVAILABLE = True
except ImportError:
    _GENJAX_AVAILABLE = False
    print("GenJAX not available — the 'Now in GenJAX' cells will not run. The numpy path works fine without it.")

---

# Problem 1: Gaussian-Gaussian Conjugate Model

We start with the **Gaussian-Gaussian** conjugate model: a Gaussian likelihood with unknown mean $\mu$ and known variance $\sigma_x^2$, with a Gaussian prior on $\mu$:

$$\mu \sim \mathcal{N}(\mu_0, \sigma_0^2) \qquad \qquad x_1, \dotsc, x_N \mid \mu, \sigma_x^2 \overset{iid}{\sim} \mathcal{N}(\mu, \sigma_x^2)$$

The conjugate posterior and posterior-predictive are:

$$\mu \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\frac{\mu_0 \sigma_0^{-2} + \sigma_x^{-2} \sum_n x_n}{\sigma_0^{-2} + N \sigma_x^{-2}},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1}\right)$$

$$x_{N+1} \mid x_1, \dotsc, x_N \sim \mathcal{N}\!\left(\text{same mean as posterior},\; \left[\sigma_0^{-2} + N \sigma_x^{-2}\right]^{-1} + \sigma_x^2\right)$$

**For Problem 1, use $\mu_0 = 0$ and $\sigma_0^2 = 1$.**


## Part 1(a): Prior plot

To provide a baseline, plot the prior distribution $p(\mu) = \mathcal{N}(\mu; \mu_0, \sigma_0^2)$ over a range that captures both tails and the peak.


In [ ]:
# SOLUTION
mu_0 = 0.0
sigma_0_squared = 1.0
mu_range = np.linspace(-4, 4, 1000)

prior_density = norm.pdf(mu_range, mu_0, np.sqrt(sigma_0_squared))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mu_range, prior_density, color='C0', lw=2, label=r'$p(\mu)$')
ax.fill_between(mu_range, prior_density, alpha=0.15, color='C0')

ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$p(\mu)$')
ax.set_title(r'Prior: $\mathcal{N}(\mu_0, \sigma_0^2)$')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


### Now in GenJAX — Part 1(a): the prior as a generative function

**Concept (Tutorial 2 Ch 2): `@gen` functions.** A GenJAX **generative function** is a Python function decorated with `@gen` that defines a *probabilistic process*. Each call to `simulate` runs the function once with a fresh random key and returns a **trace** — a record of the random choices that were made.

Below, you'll write the prior as a `@gen` function and use it to sample many values of $\mu$. Then you'll histogram the samples and overlay the analytical density. They should match (this is "Monte Carlo as sanity check").

**Key syntax:**
- `mu = normal(mu_0, sigma_0) @ "mu"` samples a normal RV and **addresses** the choice with the name `"mu"`. You can later refer to this choice by name.
- `prior.simulate(key, args)` runs the function once; `prior.simulate.vmap(in_axes=...)(keys, args)` runs it in parallel for many keys (see Ch 2).
- `trace.get_retval()` returns whatever the `@gen` function returned.

**Why bother?** For a conjugate Gaussian prior, plotting the analytical density is faster. But Monte Carlo sampling is what you'll *have* to do once the model becomes non-conjugate (Problem 2 Part (e), or any real research problem). This cell teaches the pattern.


In [ ]:
# SOLUTION — optional GenJAX path

if _GENJAX_AVAILABLE:
    # Cast model-argument scalars to float32 to match GenJAX's distribution dtype.
    mu_0 = jnp.float32(0.0)
    sigma_0 = jnp.float32(1.0)  # std, not variance, for GenJAX `normal`
    N = 5000

    @gen
    def prior(mu_0, sigma_0):
        mu = normal(mu_0, sigma_0) @ "mu"
        return mu

    keys = random.split(key, N)
    traces = jax.vmap(lambda k: prior.simulate(k, (mu_0, sigma_0)))(keys)
    mu_samples = traces.get_retval()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(np.asarray(mu_samples), bins=60, density=True, alpha=0.5,
            color='C0', label=f'simulated ({N} samples)')
    ax.plot(mu_range, norm.pdf(mu_range, mu_0, sigma_0),
            color='C3', lw=2, label='analytical prior')
    ax.set_xlabel(r'$\mu$')
    ax.set_ylabel('density')
    ax.set_title('GenJAX: simulated prior vs. analytical density')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("GenJAX not installed — skipping.")


## Part 1(b): One-datum update

Calculate and plot the **posterior** $p(\mu \mid x_1)$ **and** the **posterior-predictive** $p(x_2 \mid x_1)$ after observing $x_1 = 2$, for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$ (four distributions total).

**Question.** How does changing the likelihood variance $\sigma_x^2$ affect the posterior and the predictive? Where are the two distributions similar? Where do they differ, and why?


In [ ]:
# SOLUTION
def conjugate_update(mu_0, sigma_0_squared, sigma_x_squared, data):
    """Conjugate Gaussian-Gaussian update. Returns (post_mean, post_var, pred_var)."""
    data = np.atleast_1d(np.asarray(data, dtype=float))
    N = data.size
    sum_x = data.sum()
    posterior_precision = 1.0 / sigma_0_squared + N / sigma_x_squared
    post_mean = (mu_0 / sigma_0_squared + sum_x / sigma_x_squared) / posterior_precision
    post_var = 1.0 / posterior_precision
    pred_var = post_var + sigma_x_squared
    return post_mean, post_var, pred_var


In [ ]:
# SOLUTION
mu_0 = 0.0
sigma_0_squared = 1.0
x_1 = np.array([2.0])
sigma_x_squared_values = [0.25, 4.0]
plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    post_mean, post_var, pred_var = conjugate_update(
        mu_0, sigma_0_squared, sigma_x_squared, x_1)
    posterior = norm.pdf(plot_range, post_mean, np.sqrt(post_var))
    predictive = norm.pdf(plot_range, post_mean, np.sqrt(pred_var))

    ax.plot(plot_range, posterior, color='C0', lw=2,
            label=f'posterior  (var={post_var:.3f})')
    ax.plot(plot_range, predictive, color='C1', lw=2,
            label=f'predictive (var={pred_var:.3f})')
    ax.axvline(x_1[0], color='k', ls='--', alpha=0.5, label=r'$x_1 = 2$')

    ax.set_xlabel('value')
    ax.set_ylabel('density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared}$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


### Now in GenJAX — Part 1(b): conditioning with `generate`

> **Recommended order:** if you haven't done a GenJAX cell yet, **skip ahead to Part 2(d) "Now in GenJAX" first.** That cell uses only forward simulation (`simulate`) and is the gentlest introduction. The importance-sampling pattern in this cell is more advanced.

**Concept (Tutorial 2 Ch 4): conditioning a `@gen` function.** GenJAX gives you the posterior by *conditioning* the model on observed data. There are three ways:

1. **Rejection sampling** — simulate many traces, throw away the ones inconsistent with the observation. Simple to understand; very inefficient for continuous observations (probability of exact match = 0).
2. **`model.generate(key, constraints, args)`** — runs the model but **forces** addressed choices to take observed values, returning the trace + an importance weight.
3. **Importance sampling** — for posterior over an unobserved latent: run `generate` many times and weight each sample by its importance score.

For this cell we'll use **option 3** because the latent ($\mu$) is continuous and we want a posterior over it.

**Why bother?** For conjugate Gaussian-Gaussian, you already have the closed-form posterior — importance sampling is overkill. But this is the *exact* pattern that scales to non-conjugate models where no closed form exists. Doing it here on a problem with a known answer means you can *verify* your importance-sampling code by comparing the weighted histogram to the analytical posterior.

**Key syntax:**
- `ChoiceMap.d({"x_1": 2.0})` builds a deterministic choice map from a Python dict (matches Tutorial 2 Ch 4). The `.d` stands for "dict-constructor."
- `trace, log_weight = model.generate(key, ChoiceMap.d({"x_1": 2.0}), args)` returns the trace and the log-importance-weight.
- To turn log-weights into a weighted histogram, normalize: `weights = np.exp(log_weights - log_weights.max())`; `weights /= weights.sum()`. Then `plt.hist(mu_samples, weights=weights, density=True)`.

**On log-sum-exp normalization.** Importance weights live in *log* space because the raw probability of each trace can underflow. We subtract `log_weights.max()` to keep the largest weight at $e^0 = 1$ before exponentiating — this prevents overflow/underflow without changing the relative weights. The normalization step (`/= weights.sum()`) gives you proper probabilities.

In [ ]:
# SOLUTION — optional GenJAX path

if _GENJAX_AVAILABLE:
    # Cast model-argument scalars to float32 to match GenJAX's distribution dtype.
    mu_0 = jnp.float32(0.0)
    sigma_0 = jnp.float32(1.0)  # std

    @gen
    def gaussian_gaussian_model(mu_0, sigma_0, sigma_x):
        mu = normal(mu_0, sigma_0) @ "mu"
        x_1 = normal(mu, sigma_x) @ "x_1"
        return mu, x_1

    n_particles = 20000
    plot_range = np.linspace(-4, 6, 1000)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sigma_x_squared in zip(axes, [0.25, 4.0]):
        sigma_x = jnp.float32(np.sqrt(sigma_x_squared))
        constraints = ChoiceMap.d({"x_1": jnp.float32(2.0)})

        keys = random.split(key, n_particles)

        def one(k, sx=sigma_x):
            trace, log_w = gaussian_gaussian_model.generate(
                k, constraints, (mu_0, sigma_0, sx))
            return trace.get_choices()["mu"], log_w

        mu_s, log_w = jax.vmap(one)(keys)
        w = np.exp(np.asarray(log_w) - np.max(np.asarray(log_w)))
        w = w / w.sum()

        # Closed-form posterior for comparison.
        pm, pv, _ = conjugate_update(mu_0, sigma_0**2, sigma_x_squared, np.array([2.0]))

        ax.hist(np.asarray(mu_s), bins=60, weights=w, density=True,
                alpha=0.5, color='C0', label='importance-sampled posterior')
        ax.plot(plot_range, norm.pdf(plot_range, pm, np.sqrt(pv)),
                color='C3', lw=2, label='closed-form posterior')
        ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared}$')
        ax.set_xlabel(r'$\mu$')
        ax.set_ylabel('density')
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 1(b)).**

The posterior and predictive are always centered at the same posterior mean,
but the predictive is wider by exactly $\sigma_x^2$ — the irreducible
per-observation noise.

Increasing $\sigma_x^2$ from 0.25 to 4 makes the posterior move *less* toward
$x_1 = 2$ (a diffuse likelihood lets the prior dominate) but makes the
predictive *much wider* (the variance gain $\sigma_x^2$ is large). So the
likelihood variance has opposite-direction effects on the two distributions.


## Part 1(c): Multiple-datum update

Now observe five data points: $(x_1, \dotsc, x_5) = (2.1, 2.5, 1.4, 2.2, 1.8)$. Plot the posterior and predictive for $\sigma_x^2 = 0.25$ and $\sigma_x^2 = 4$.

The average of these five points is exactly $2.0$, same as the single datum in Part 1(b). **Compare** the resulting posteriors and predictives to Part 1(b).


In [ ]:
# SOLUTION
mu_0 = 0.0
sigma_0_squared = 1.0
data = np.array([2.1, 2.5, 1.4, 2.2, 1.8])
sigma_x_squared_values = [0.25, 4.0]
plot_range = np.linspace(-4, 6, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sigma_x_squared in zip(axes, sigma_x_squared_values):
    post_mean, post_var, pred_var = conjugate_update(
        mu_0, sigma_0_squared, sigma_x_squared, data)
    posterior = norm.pdf(plot_range, post_mean, np.sqrt(post_var))
    predictive = norm.pdf(plot_range, post_mean, np.sqrt(pred_var))

    ax.plot(plot_range, posterior, color='C0', lw=2,
            label=f'posterior  (var={post_var:.3f})')
    ax.plot(plot_range, predictive, color='C1', lw=2,
            label=f'predictive (var={pred_var:.3f})')
    ax.axvline(data.mean(), color='k', ls='--', alpha=0.5,
               label=f'data mean = {data.mean():.1f}')

    ax.set_xlabel('value')
    ax.set_ylabel('density')
    ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared},\ N = 5$')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


### Now in GenJAX — Part 1(c): conditioning on multiple observations

**Concept extension (Tutorial 2 Ch 4): multiple observations in one choice map.** The same `generate` API handles many observations at once — just put them all into the choice map.

**Two ways to express N observations:**

1. **Plate notation in the model.** Inside the `@gen` function, sample N observations addressed as `"x_1"`, `"x_2"`, ..., `"x_N"`. Then your choice map is `{"x_1": 2.1, "x_2": 2.5, ..., "x_5": 1.8}`. Easy to read; verbose.

2. **Vectorized addressing (Tutorial 2 Ch 6).** Use `jax.vmap` over a single `"x"` address so the model produces an array of observations under one address. More concise; requires understanding `vmap`.

For this cell, **use approach 1** (it's clearer when you're first learning the pattern). Approach 2 is a Tutorial 2 Ch 6 topic.


In [ ]:
# SOLUTION — optional GenJAX path

if _GENJAX_AVAILABLE:
    # Cast model-argument scalars to float32 to match GenJAX's distribution dtype.
    mu_0 = jnp.float32(0.0)
    sigma_0 = jnp.float32(1.0)
    data_5 = [2.1, 2.5, 1.4, 2.2, 1.8]

    @gen
    def gaussian_gaussian_5(mu_0, sigma_0, sigma_x):
        mu = normal(mu_0, sigma_0) @ "mu"
        x_1 = normal(mu, sigma_x) @ "x_1"
        x_2 = normal(mu, sigma_x) @ "x_2"
        x_3 = normal(mu, sigma_x) @ "x_3"
        x_4 = normal(mu, sigma_x) @ "x_4"
        x_5 = normal(mu, sigma_x) @ "x_5"
        return mu

    n_particles = 20000
    plot_range = np.linspace(-4, 6, 1000)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sigma_x_squared in zip(axes, [0.25, 4.0]):
        sigma_x = jnp.float32(np.sqrt(sigma_x_squared))
        constraints = ChoiceMap.d({
            k: jnp.float32(v) for k, v in
            {"x_1": 2.1, "x_2": 2.5, "x_3": 1.4, "x_4": 2.2, "x_5": 1.8}.items()
        })
        keys = random.split(key, n_particles)

        def one(k, sx=sigma_x):
            trace, log_w = gaussian_gaussian_5.generate(
                k, constraints, (mu_0, sigma_0, sx))
            return trace.get_choices()["mu"], log_w

        mu_s, log_w = jax.vmap(one)(keys)
        w = np.exp(np.asarray(log_w) - np.max(np.asarray(log_w)))
        w = w / w.sum()

        pm, pv, _ = conjugate_update(
            mu_0, sigma_0**2, sigma_x_squared, np.array(data_5))

        ax.hist(np.asarray(mu_s), bins=60, weights=w, density=True,
                alpha=0.5, color='C0', label='importance-sampled posterior')
        ax.plot(plot_range, norm.pdf(plot_range, pm, np.sqrt(pv)),
                color='C3', lw=2, label='closed-form posterior')
        ax.set_title(rf'$\sigma_x^2 = {sigma_x_squared},\ N = 5$')
        ax.set_xlabel(r'$\mu$')
        ax.set_ylabel('density')
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 1(c)).**

The data mean is 2.0 in both Part 1(b) and 1(c), so the posterior *means* are
similar. What changes is the *variance*: with $N = 5$ the posterior precision
$1/\sigma_0^2 + N/\sigma_x^2$ has a five-times-larger data term, so the
posterior is markedly tighter — dramatically so for $\sigma_x^2 = 0.25$
(variance $\approx 0.048$). The predictive variance is still dominated by
$\sigma_x^2$: it asymptotes to $\sigma_x^2$ no matter how much data we collect.


---

# Problem 2: Gaussian Mixture / Categorization

In this problem, we make **categorization decisions** for two categories, each defined as a Gaussian distribution. You will derive the probability of an item being in one category vs. the other, then explore how the variances and prior probability of each category affect the posterior and the predictive distribution.

Data are generated by first picking which of two categories $c = 1, 2$ a datum belongs to (according to their prior probability) and then generating the datum from the corresponding category's likelihood:

$$c_n | \theta \sim \text{Bernoulli}(\theta) \qquad \qquad x_n | \mu_{c(n)}, \sigma_{c(n)}^2 \overset{iid}{\sim} \mathcal{N}(\mu_{c(n)}, \sigma_{c(n)}^2)$$

$c_n = 1$ with probability $\theta$ (and $c_n = 2$ with probability $1 - \theta$), so the prior probability of category 1 is $\theta$: $P(c_n = 1) = \theta$.

**For all of Problem 2, assume $\mu_1 = -1$ and $\mu_2 = 1$.**


## Part 2(a): Derivation — Categorization

Using **Bayes' rule**, derive the probability of a single datum being in category 1: $P(c_1 = 1 | x_1)$. You can assume that the values of $\mu_1, \mu_2, \sigma_1^2,$ and $\sigma_2^2$ are given parameters. Show your work (handwritten derivation scanned in as an image is fine).

As the next problem depends on this answer, the derivation should end up with:

$$P(c_1 = 1 | x_1) = \frac{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}{\theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)}$$

**Grading note: no credit for transcribing the given answer. Show every step (Bayes' rule, expand the denominator with the Law of Total Probability).**

#### **Derivation**

By **Bayes' rule** with prior $P(c_1 = 1) = \theta$ and likelihood
$p(x_1 \mid c_1 = 1) = \mathcal{N}(x_1; \mu_1, \sigma_1^2)$:

$$P(c_1 = 1 \mid x_1) = \frac{\theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}{p(x_1)}.$$

The denominator follows from the **Law of Total Probability**:

$$p(x_1) = \theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)
+ (1-\theta)\, \mathcal{N}(x_1; \mu_2, \sigma_2^2).$$

Substituting:

$$P(c_1 = 1 \mid x_1) = \frac{\theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)}
{\theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)
+ (1-\theta)\, \mathcal{N}(x_1; \mu_2, \sigma_2^2)}.$$


## Part 2(b): Categorization

Calculate and plot $P(c_1 = 1 | x_1)$ for:

1. $\theta = 0.5$ and $\theta = 0.75$, with $\sigma_1^2 = \sigma_2^2 = 1$.
2. $\theta = 0.5$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.
3. $\theta = 0.75$ when $\sigma_1^2 = 0.5$ and $\sigma_2^2 = 2$.

**Question.** Describe the effect of changing the prior and the variance on categorization decisions. Do they have the same effect? Why or why not?


In [ ]:
# SOLUTION
mu_1, mu_2 = -1.0, 1.0
theta_values = [0.5, 0.75]
configs = [(1, 1), (0.5, 2)]
colors = {0.5: "C0", 0.75: "C1"}
linestyles = {(1, 1): "-", (0.5, 2): "--"}
x_range = np.linspace(-6, 6, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

for theta in theta_values:
    for sigma_1_sq, sigma_2_sq in configs:
        lik1 = norm.pdf(x_range, mu_1, np.sqrt(sigma_1_sq))
        lik2 = norm.pdf(x_range, mu_2, np.sqrt(sigma_2_sq))
        posterior_c1 = (theta * lik1) / (theta * lik1 + (1 - theta) * lik2)
        ax.plot(x_range, posterior_c1,
                color=colors[theta],
                linestyle=linestyles[(sigma_1_sq, sigma_2_sq)],
                lw=2,
                label=rf'$\theta={theta},\ \sigma_1^2={sigma_1_sq},\ \sigma_2^2={sigma_2_sq}$')

ax.axhline(0.5, color='k', alpha=0.3, lw=1)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$P(c_1 = 1 | x)$')
ax.set_title('Posterior categorization probability')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


### Now in GenJAX — Part 2(b): the mixture model + importance-sampling for $P(c=1|x)$

> **Recommended order:** if you haven't done a GenJAX cell yet, **skip ahead to Part 2(d) "Now in GenJAX" first.** That cell uses only forward simulation (`simulate`) and is the gentlest introduction to the framework. Come back here once you're comfortable.

**Concept (Tutorial 2 Ch 4): conditioning a *discrete* latent.** Here the unknown is $c \in \{1, 2\}$ (category). The posterior $P(c=1|x_1)$ is a discrete probability — one number, not a distribution over $\mu$.

**Two ways to get it in GenJAX:**

1. **`generate` + importance.** Build a choice map `{"observation": x_1}`, run `generate` many times, compute $P(c=1|x_1)$ as the importance-weighted fraction of traces where `"category" == 1`.
2. **Rejection sampling.** Simulate many traces unconditionally, keep only those whose `"observation"` is close to $x_1$, count what fraction have `"category" == 1`. **Inefficient** because $x_1$ is continuous, but conceptually transparent.

We'll use **approach 1** here.

**`flip` returns Boolean.** GenJAX `flip(theta)` returns `True`/`False`. Cast via `jnp.where(c, mu_1, mu_2)` to pick the corresponding component mean.

**Key syntax preview** (you'll use the same model in Part 2(d)):
- `c = flip(theta) @ "category"` — discrete choice; **`flip` takes a probability, not a logit**
- `mu_c = jnp.where(c, mu_1, mu_2)` — pick component mean by category
- `x = normal(mu_c, sigma_c) @ "observation"` — likelihood
- `ChoiceMap.d({"observation": x_1})` — the `.d` constructor builds a deterministic choice map from a dict (matches Tutorial 2 Ch 4)

In [ ]:
# SOLUTION — optional GenJAX path

if _GENJAX_AVAILABLE:
    @gen
    def mixture_model(theta, mu_1, mu_2, sigma_1, sigma_2):
        c = flip(theta) @ "category"
        mu_c = jnp.where(c, mu_1, mu_2)
        sigma_c = jnp.where(c, sigma_1, sigma_2)
        x = normal(mu_c, sigma_c) @ "observation"
        return x, c

    # One configuration: theta=0.5, equal variances.
    # Cast model-argument scalars to float32 to match GenJAX's distribution dtype.
    theta = jnp.float32(0.5)
    mu_1, mu_2 = jnp.float32(-1.0), jnp.float32(1.0)
    sigma_1, sigma_2 = jnp.float32(1.0), jnp.float32(1.0)
    n_particles = 4000
    x_grid = np.linspace(-6, 6, 40)

    def p_c1_given_x(x_obs):
        constraints = ChoiceMap.d({"observation": jnp.float32(x_obs)})
        keys = random.split(random.PRNGKey(int(abs(x_obs) * 1000) + 1), n_particles)

        def one(k):
            trace, log_w = mixture_model.generate(
                k, constraints, (theta, mu_1, mu_2, sigma_1, sigma_2))
            return trace.get_choices()["category"], log_w

        cats, log_w = jax.vmap(one)(keys)
        w = np.exp(np.asarray(log_w) - np.max(np.asarray(log_w)))
        w = w / w.sum()
        return float(np.sum(w * np.asarray(cats)))

    empirical = np.array([p_c1_given_x(x) for x in x_grid])

    # Analytical curve for comparison.
    lik1 = norm.pdf(x_grid, mu_1, sigma_1)
    lik2 = norm.pdf(x_grid, mu_2, sigma_2)
    analytical = (theta * lik1) / (theta * lik1 + (1 - theta) * lik2)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x_grid, empirical, 'o', color='C0', label='importance-sampled P(c=1|x)')
    ax.plot(x_grid, analytical, color='C3', lw=2, label='analytical P(c=1|x)')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$P(c_1 = 1 | x)$')
    ax.set_title('GenJAX importance sampling vs. analytical')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 2(b)).**

The prior $\theta$ and the variances act differently. $\theta$ adds a constant
offset to the log-odds — it shifts the whole curve up/down (and with equal
variances, moves the decision boundary) without changing its shape. The
variances change the *shape*: when $\sigma_1^2 \neq \sigma_2^2$ the log-odds is
quadratic in $x$, so the curve is no longer a simple monotone sigmoid. They are
not the same effect.


## Part 2(c): Derivation — Prediction

Using Bayes' rule and the **Law of Total Probability**, derive $p(x_1)$ for this model (without any given data). As the next problem depends on this answer, the derivation should end up with:

$$p(x_1) = \theta \, \mathcal{N}(x_1; \mu_1, \sigma_1^2) + (1 - \theta) \, \mathcal{N}(x_1; \mu_2, \sigma_2^2)$$

**Grading note: no credit for transcribing the given answer. Show every step (Law of Total Probability over $c$, then expand $p(x|c)P(c)$ for each category).**

#### **Derivation**

By the **Law of Total Probability**, summing the joint over the two categories:

$$p(x_1) = \sum_c p(x_1 \mid c)\, P(c)
= \theta\, \mathcal{N}(x_1; \mu_1, \sigma_1^2)
+ (1-\theta)\, \mathcal{N}(x_1; \mu_2, \sigma_2^2).$$


## Part 2(d): Prediction

Plot $p(x_1)$ for the same configurations as in Part 2(b).

This $p(x_1)$ is sometimes called the *marginal data distribution*; this model is called a **mixture model** because it composes a new distribution by mixing two (or more) component distributions.

**Question.** How does the prior and variance affect $p(x_1)$? Do they have the same effect? Why or why not?


In [ ]:
# SOLUTION
mu_1, mu_2 = -1.0, 1.0
theta_values = [0.5, 0.75]
configs = [(1, 1), (0.5, 2)]
colors = {0.5: "C0", 0.75: "C1"}
linestyles = {(1, 1): "-", (0.5, 2): "--"}
x_range = np.linspace(-6, 6, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

for theta in theta_values:
    for sigma_1_sq, sigma_2_sq in configs:
        p_x = (theta * norm.pdf(x_range, mu_1, np.sqrt(sigma_1_sq))
               + (1 - theta) * norm.pdf(x_range, mu_2, np.sqrt(sigma_2_sq)))
        ax.plot(x_range, p_x,
                color=colors[theta],
                linestyle=linestyles[(sigma_1_sq, sigma_2_sq)],
                lw=2,
                label=rf'$\theta={theta},\ \sigma_1^2={sigma_1_sq},\ \sigma_2^2={sigma_2_sq}$')

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$p(x)$')
ax.set_title('Marginal (predictive) distribution')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


### Now in GenJAX — Part 2(d): sampling the marginal $p(x)$

**Concept (Tutorial 2 Ch 2): forward simulation = sampling from the marginal.** If you simulate from the mixture model *without* conditioning on anything, the distribution of observed `x` values **is** the marginal $p(x)$. This is the simplest GenJAX cell in the assignment — no choice map, no importance sampling, just `simulate`.

**Key syntax:**
- `traces = jax.vmap(lambda k: mixture_model.simulate(k, args))(keys)` — N parallel simulations
- `x_samples = traces.get_choices()["observation"]` — pull the observed-x value out of each trace
- `plt.hist(x_samples, bins=80, density=True)` — empirical marginal


In [ ]:
# SOLUTION — optional GenJAX path

if _GENJAX_AVAILABLE:
    @gen
    def mixture_model(theta, mu_1, mu_2, sigma_1, sigma_2):
        c = flip(theta) @ "category"
        mu_c = jnp.where(c, mu_1, mu_2)
        sigma_c = jnp.where(c, sigma_1, sigma_2)
        x = normal(mu_c, sigma_c) @ "observation"
        return x, c

    # Cast model-argument scalars to float32 to match GenJAX's distribution dtype.
    theta = jnp.float32(0.75)
    mu_1, mu_2 = jnp.float32(-1.0), jnp.float32(1.0)
    sigma_1, sigma_2 = jnp.float32(1.0), jnp.float32(1.0)
    N = 5000

    keys = random.split(key, N)
    traces = jax.vmap(
        lambda k: mixture_model.simulate(k, (theta, mu_1, mu_2, sigma_1, sigma_2))
    )(keys)
    x_samples, _ = traces.get_retval()

    x_grid = np.linspace(-6, 6, 1000)
    analytical = (theta * norm.pdf(x_grid, mu_1, sigma_1)
                  + (1 - theta) * norm.pdf(x_grid, mu_2, sigma_2))

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(np.asarray(x_samples), bins=60, density=True, alpha=0.5,
            color='C0', label=f'simulated ({N} samples)')
    ax.plot(x_grid, analytical, color='C3', lw=2, label='analytical p(x)')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel('density')
    ax.set_title('GenJAX forward simulation vs. analytical marginal')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("GenJAX not installed — skipping.")


**Your answer (Part 2(d)).**

$\theta$ controls the relative *heights* of the two component peaks; the
variances control their *widths*. With equal $\sigma^2$ the mixture is cleanly
bimodal; with $(\sigma_1^2, \sigma_2^2) = (0.5, 2)$ one component becomes a
narrow spike and the other a broad shoulder, and the visible bimodality can
wash out. They are not the same effect — $\theta$ redistributes mass *between*
components, the variances redistribute each component's mass *across $x$*.


---

## Submission

Submit this completed notebook (runs end-to-end with no errors) plus your derivations for Parts 2(a) and 2(c) (either inline as LaTeX or as scanned/typeset images).

**A note on Part 2(e).** The canonical (GenJAX) stencil has an additional Part 2(e) that asks students to build a `@gen` mixture model and verify the marginal $p(x)$ by Monte Carlo simulation. **For the Python (no-GenJAX) and R paths, Part 2(e) is bonus, not required.** Two ways to earn the bonus credit:

1. **Numpy-only forward simulation (easiest).** Add a cell below that samples $N=2000$ values: for each draw, flip a coin with probability $\theta=0.7$ to pick a category, then sample $x$ from the corresponding Gaussian. Histogram the samples and overlay the analytical $p(x)$ from Part 2(d). A 5-line `np.random.binomial` / `np.random.normal` solution counts.
2. **The "Now in GenJAX" cells.** If you completed cell 33 (Part 2(d) forward simulation) and/or cell 26 (Part 2(b) importance sampling), you've already done the equivalent. Mention this in your submission.

Either path earns the bonus. The numpy version is recommended if you don't want to install GenJAX.